In [13]:
import pandas as pd
import numpy as np
from datetime import datetime


In [14]:
df=pd.read_csv("rss_articles.csv")

In [24]:
df.head()


,source,title,text_body,Time,Date,sentiment_finbert,sentiment_label_finbert,sentiment_score_finbert,sentiment_numeric_finbert
0,ECONOMIC TIMES,Dalal Street Week Ahead: Technical charts sign...,"Indian markets traded rangebound last week, en...",01-November-25,2025-11-01,"{'label': 'Positive', 'score': 0.9991878867149...",positive,0.999188,1
1,ECONOMIC TIMES,F&amp;O Talk| Nifty logs 11 sessions of tight ...,Markets ended lower four-week rally due profit...,01-November-25,2025-11-01,"{'label': 'Positive', 'score': 0.9999996423721...",positive,1.000000,1
2,ECONOMIC TIMES,"CarTrade Tech, Chennai Petro among 10 smallcap...","Markets ended four-week winning streak, closin...",01-November-25,2025-11-01,"{'label': 'Positive', 'score': 1.0}",positive,1.000000,1
3,ECONOMIC TIMES,"CarTrade Tech, Chennai Petro among 10 smallcap...","Amid volatility, broader indices continued out...",01-November-25,2025-11-01,"{'label': 'Positive', 'score': 0.9999997615814...",positive,1.000000,1
4,ECONOMIC TIMES,Sectoral and thematic mutual funds outperform ...,"Motilal Oswal Nasdaq 100 FOF, topper list inte...",01-November-25,2025-11-01,"{'label': 'Neutral', 'score': 0.9999977350234985}",neutral,0.000000,0


In [25]:
df.tail()

,source,title,text_body,Time,Date,sentiment_finbert,sentiment_label_finbert,sentiment_score_finbert,sentiment_numeric_finbert
437,HINDUSTAN TIMES - BUSINESS,Big Tech banks on artificial intelligence to d...,"become ‘future ready’, Indian companies starte...",31-October-25,2025-10-31,"{'label': 'Neutral', 'score': 0.9996129870414734}",neutral,0.0,0
438,HINDUSTAN TIMES - BUSINESS,Starlink jobs in India: Starlink starts on-gro...,Starlink jobs India Bengaluru finance & accoun...,31-October-25,2025-10-31,"{'label': 'Neutral', 'score': 0.9999969005584717}",neutral,0.0,0
439,HINDUSTAN TIMES - BUSINESS,"Lenskart IPO GMP: From RHP to IPO day, what th...",trajectory Lenskart IPO GMP shows exuberance g...,31-October-25,2025-10-31,"{'label': 'Neutral', 'score': 0.994719386100769}",neutral,0.0,0
440,HINDUSTAN TIMES - BUSINESS,"Lenskart IPO: GMP to IPO price and valuation, ...",Also focus co-founder Peyush Bansal net worth ...,31-October-25,2025-10-31,"{'label': 'Neutral', 'score': 0.9999990463256836}",neutral,0.0,0
441,HINDUSTAN TIMES - BUSINESS,Canva’s Creative OS and Affinity app lay found...,Melanie Perkins’ vision transitioning eras beg...,30-October-25,2025-10-30,"{'label': 'Neutral', 'score': 0.5841019153594971}",neutral,0.0,0


In [16]:
df = df.drop(columns= ['author','url'])

In [17]:
df["Time"] = pd.to_datetime(df["timestamp"]).dt.strftime("%d-%B-%y")
df["Date"] = pd.to_datetime(df["timestamp"]).dt.strftime("%Y-%m-%d")


In [18]:
df = df.drop(columns=['timestamp'])

In [19]:
df

,source,title,text_body,Time,Date
0,ECONOMIC TIMES,Dalal Street Week Ahead: Technical charts sign...,"Indian markets traded rangebound last week, en...",01-November-25,2025-11-01
1,ECONOMIC TIMES,F&amp;O Talk| Nifty logs 11 sessions of tight ...,Markets ended lower after a four-week rally du...,01-November-25,2025-11-01
2,ECONOMIC TIMES,"CarTrade Tech, Chennai Petro among 10 smallcap...","Markets ended their four-week winning streak, ...",01-November-25,2025-11-01
3,ECONOMIC TIMES,"CarTrade Tech, Chennai Petro among 10 smallcap...","Amid the volatility, the broader indices conti...",01-November-25,2025-11-01
4,ECONOMIC TIMES,Sectoral and thematic mutual funds outperform ...,"Motilal Oswal Nasdaq 100 FOF, the topper in th...",01-November-25,2025-11-01
...,...,...,...,...,...
437,HINDUSTAN TIMES - BUSINESS,Big Tech banks on artificial intelligence to d...,"To become ‘future ready’, Indian companies hav...",31-October-25,2025-10-31
438,HINDUSTAN TIMES - BUSINESS,Starlink jobs in India: Starlink starts on-gro...,All the Starlink jobs in India are in Bengalur...,31-October-25,2025-10-31
439,HINDUSTAN TIMES - BUSINESS,"Lenskart IPO GMP: From RHP to IPO day, what th...",The trajectory of the Lenskart IPO GMP shows h...,31-October-25,2025-10-31
440,HINDUSTAN TIMES - BUSINESS,"Lenskart IPO: GMP to IPO price and valuation, ...",Also in focus is co-founder Peyush Bansal and ...,31-October-25,2025-10-31


In [20]:
from transformers import BertTokenizer, BertForSequenceClassification
from tqdm import tqdm
from transformers import pipeline
from nltk.corpus import stopwords
import nltk

tokenizer = BertTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = BertForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")
finbert = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clear_text(text):
  words = str(text).split()
  filtered_words = [word for word in words if word.lower() not in stop_words]
  return " ".join(filtered_words)[:512]

df['text_body'] = df['text_body'].apply(clear_text)

tqdm.pandas()
df['sentiment_finbert'] = df['text_body'].progress_apply(lambda x: finbert(x)[0])

Device set to use cpu
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Amogh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
100%|██████████| 442/442 [01:38<00:00,  4.48it/s]


In [21]:
df['sentiment_label_finbert'] = df['sentiment_finbert'].apply(lambda x: x['label'].lower())
df['sentiment_score_finbert'] = df['sentiment_finbert'].apply(lambda x: x['score'])
sentiment_mapping = {"neutral": 0, "positive": 1, "negative": -1}
df['sentiment_numeric_finbert'] = df['sentiment_label_finbert'].map(sentiment_mapping)
df["sentiment_score_finbert"] = df["sentiment_numeric_finbert"]*df["sentiment_score_finbert"]

In [22]:
df_ecotime = df[0:76]
df_ecomarket = df[76:126]
df_ecostock = df[126:176]
df_ecoipos = df[176:226]
df_ecoexpert = df[226:276]
df_ecobonds = df[276:326]
df_moneycontrol = df[326:341]
df_mintmoney=df[341:376]
df_mintmarket = df[376:411]
df_timesindia = df[411:431]
df_hindustan = df[431:442]

In [23]:
df_ecotime

,source,title,text_body,Time,Date,sentiment_finbert,sentiment_label_finbert,sentiment_score_finbert,sentiment_numeric_finbert
0,ECONOMIC TIMES,Dalal Street Week Ahead: Technical charts sign...,"Indian markets traded rangebound last week, en...",01-November-25,2025-11-01,"{'label': 'Positive', 'score': 0.9991878867149...",positive,0.999188,1
1,ECONOMIC TIMES,F&amp;O Talk| Nifty logs 11 sessions of tight ...,Markets ended lower four-week rally due profit...,01-November-25,2025-11-01,"{'label': 'Positive', 'score': 0.9999996423721...",positive,1.000000,1
2,ECONOMIC TIMES,"CarTrade Tech, Chennai Petro among 10 smallcap...","Markets ended four-week winning streak, closin...",01-November-25,2025-11-01,"{'label': 'Positive', 'score': 1.0}",positive,1.000000,1
3,ECONOMIC TIMES,"CarTrade Tech, Chennai Petro among 10 smallcap...","Amid volatility, broader indices continued out...",01-November-25,2025-11-01,"{'label': 'Positive', 'score': 0.9999997615814...",positive,1.000000,1
4,ECONOMIC TIMES,Sectoral and thematic mutual funds outperform ...,"Motilal Oswal Nasdaq 100 FOF, topper list inte...",01-November-25,2025-11-01,"{'label': 'Neutral', 'score': 0.9999977350234985}",neutral,0.000000,0
...,...,...,...,...,...,...,...,...,...
71,ECONOMIC TIMES,"DB Realty, Unitech stocks rose",stocks DB Realty Unitech rose top executives c...,30-November-11,2011-11-30,"{'label': 'Neutral', 'score': 0.9999885559082031}",neutral,0.000000,0
72,ECONOMIC TIMES,Fitch withdraws Reliance Capital ratings,Fitch Friday said withdrawn ratings Reliance C...,20-October-11,2011-10-20,"{'label': 'Neutral', 'score': 0.9990068078041077}",neutral,0.000000,0
73,ECONOMIC TIMES,'Discoms' poor fin health poses risks for trad...,poor financial health state electricity boards...,20-October-11,2011-10-20,"{'label': 'Negative', 'score': 0.9999972581863...",negative,-0.999997,-1
74,ECONOMIC TIMES,Cigarette cos shares outperform benchmark Sensex,Shares cigarette companies rallied past one mo...,19-July-11,2011-07-19,"{'label': 'Positive', 'score': 0.9684965014457...",positive,0.968497,1
